# Lesson 85 · A KV cache from scratch — why ChatGPT speeds up as it answers: updating the key/value cache, O(seq²) → O(seq)

**Learning goals**
- See exactly which repeated computation a KV cache saves
- Implement scaled dot-product attention (pure NumPy)
- Implement a KVCache class: append + retrieve
- Compare inference speed with and without a KV cache

> **Scope**: a KV cache saves the repeated work of the K/V projections, not attention itself. At step t the new token still has to look at the whole history, so the formula is the same as in L83, with one extra "store / fetch" step for the cache; Q isn't cached either, because each step has only one new token.

← **Previous lesson** · [L84 · LoRA](L84_lora.ipynb)

> Last lesson covered **LoRA**: W = W₀ + BA, fine-tuning a GPT-style model with 0.1 % of the parameters.
> This lesson explores **a KV cache from scratch**.

## Why does ChatGPT speed up as it answers?

While ChatGPT generates its answer to you word by word, at every step it does one thing: read **all the tokens so far**, then predict the next one.

**The naive way**: every time a new token is generated, re-project all the earlier tokens into K and V. Step 1 does it once, step 100 does it 100 times, and summed up that is 1+2+…+T = O(T²) matrix multiplications.

**The KV-cache idea**: keep a memo — **store** the K and V computed at each step **in a cache**. Step t+1 then only has to compute K and V for the **new token** and append them to the end of the cache.

```
step t (no cache):   K/V projection of t tokens → O(t·d²)
step t (with cache): K/V projection of 1 token, append → O(d²)

total cost over the sequence:
  no cache   = 1+2+…+T = O(T²·d²)
  with cache = T × 1   = O(T·d²)      ← saves a factor of order T (≈ T/2)
```

**The memory cost**: cache size = T × n_layers × n_heads × head_dim. For long sequences video memory is the bottleneck — which is why LLM inference frameworks (vLLM, PagedAttention) manage this memory so carefully.

This section's task: implement a `KVCache` class in pure NumPy and verify that the output is exactly the same with and without the cache.

## ⚠️ Common misconceptions

> Don't think of the KV cache as "trading accuracy for speed". It is an **exactly lossless** optimization — the output with and without the cache is equal value for value (this lesson checks it yourself with `atol=1e-9`). It just **stores and reuses** the repeated K/V projections; not a single number changes.
>
> Another easy mistake: what gets cached is **K and V**, not Q. Q is only ever "the one new token right now"; past tokens never need to ask their question again — and that is exactly why each step computes only 1 token.

In [ ]:
import numpy as np
import time

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """Scaled dot-product attention, pure NumPy.

    Args:
        Q: (n_heads, seq_q, head_dim)
        K: (n_heads, seq_k, head_dim)
        V: (n_heads, seq_k, head_dim)
        mask: optional (seq_q, seq_k) bool mask, True=mask out

    Returns:
        (n_heads, seq_q, head_dim)
    """
    # ✏️ TODO (4 steps):
    # 1) scores = Q @ K.swapaxes(-1, -2) / np.sqrt(head_dim)   → (n_heads, seq_q, seq_k)
    # 2) if mask is not None: scores[..., mask] = -np.inf       # the mask acts on the last two axes (seq_q, seq_k)
    # 3) weights = softmax(scores, axis=-1)                     → each row sums to 1
    # 4) return weights @ V                                     → (n_heads, seq_q, head_dim)
    raise NotImplementedError("TODO")

# Check (skipped while unimplemented, so Run All isn't interrupted)
np.random.seed(0)
n_heads, seq, d = 2, 4, 8
Q = np.random.randn(n_heads, seq, d)
K = np.random.randn(n_heads, seq, d)
V = np.random.randn(n_heads, seq, d)
try:
    out = scaled_dot_product_attention(Q, K, V)
    assert out.shape == (n_heads, seq, d)
    print(f'attention output shape: {out.shape}  ✅')
except (NotImplementedError, TypeError):
    print('⬜ scaled_dot_product_attention not implemented yet; finish the 4 TODO steps first')

## ✏️ Implement `KVCache`

**Data structure**: for each layer (the layer key) keep two matrices of shape `(n_heads, seq_so_far, head_dim)`, which grow with each append.

**Signature**:
```python
class KVCache:
    def update(self, layer, new_k, new_v):
        # new_k/new_v: (n_heads, 1, head_dim)
        # returns: (k_full, v_full) — (n_heads, seq_so_far, head_dim)
```

**Two steps to implement it**:

| Step | Action | Key point |
|---|---|---|
| 1 | First call (`layer not in self._k`) | assign directly: `self._k[layer] = new_k` |
| 2 | Later calls | `np.concatenate([self._k[layer], new_k], axis=1)`; axis=1 is the seq axis |

**Acceptance criteria**:
- `cache.seq_len()` goes up by 1 after each call
- the token-by-token attention output with the cache is exactly the same as a single full computation with a causal mask (atol=1e-9)

In [ ]:
class KVCache:
    """KV cache: each Transformer layer stores its past K, V and appends step by step."""

    def __init__(self, n_heads: int, head_dim: int):
        self.n_heads = n_heads
        self.head_dim = head_dim
        self._k = {}   # layer → (n_heads, seq, head_dim)
        self._v = {}

    def update(self, layer, new_k, new_v):
        """Append the new K/V and return the full cache. new_k/new_v: (n_heads, 1, head_dim)"""
        # ✏️ TODO:
        # if layer not in self._k:            # first call: assign directly
        #     self._k[layer] = new_k
        #     self._v[layer] = new_v
        # else:                               # later calls: concatenate along the seq axis
        #     self._k[layer] = np.concatenate([self._k[layer], new_k], axis=1)
        #     self._v[layer] = np.concatenate([self._v[layer], new_v], axis=1)
        # return self._k[layer], self._v[layer]
        raise NotImplementedError("TODO")

    def seq_len(self):
        if not self._k: return 0
        return next(iter(self._k.values())).shape[1]

# You can also import it directly from aurora.llm:
# from aurora.llm import KVCache

In [ ]:
# Check: attention output with and without the cache must match exactly (causal masks aligned)
n_heads, total_seq, d = 2, 6, 8
np.random.seed(42)

K_all = np.random.randn(n_heads, total_seq, d)
V_all = np.random.randn(n_heads, total_seq, d)
Q_all = np.random.randn(n_heads, total_seq, d)

# No cache: one full computation with a causal mask
# mask[i, j]=True means mask out (hide future positions j > i)
causal_mask = ~np.tril(np.ones((total_seq, total_seq), dtype=bool))

try:
    out_no_cache = scaled_dot_product_attention(Q_all, K_all, V_all, mask=causal_mask)
    # Numerical check: the attention weights of query 0 over all keys should sum to 1
    _scores_check = Q_all[:, :1, :] @ K_all.swapaxes(-1, -2) / np.sqrt(Q_all.shape[-1])
    _w_check = np.exp(_scores_check - _scores_check.max(axis=-1, keepdims=True))
    _w_check /= _w_check.sum(axis=-1, keepdims=True)
    assert np.allclose(_w_check.sum(axis=-1), 1.0, atol=1e-6), "weight rows should sum to 1"

    # With cache: append token by token, same parameters and test data
    cache = KVCache(n_heads=n_heads, head_dim=d)
    out_with_cache = np.zeros_like(Q_all)
    for t in range(total_seq):
        new_k = K_all[:, t:t+1, :]
        new_v = V_all[:, t:t+1, :]
        new_q = Q_all[:, t:t+1, :]
        k_full, v_full = cache.update(0, new_k, new_v)
        out_t = scaled_dot_product_attention(new_q, k_full, v_full)
        out_with_cache[:, t:t+1, :] = out_t

    # Compare: every token's output should be identical in both
    assert np.allclose(out_no_cache, out_with_cache, atol=1e-9), (
        f'results with and without the cache differ!\n'
        f'max diff = {np.abs(out_no_cache - out_with_cache).max():.2e}\n'
        f'hint: check the causal-mask logic in scaled_dot_product_attention'
    )
    print(f'output shape without cache: {out_no_cache.shape}')
    print(f'output shape with cache: {out_with_cache.shape}')
    print(f'cache length (tokens): {cache.seq_len()}')
    print('✅ with cache == without cache (causal masks aligned): the KV cache is correct')

except (NotImplementedError, TypeError):
    print('⬜ scaled_dot_product_attention or KVCache.update not implemented yet')


## Speed comparison

In [ ]:
# Demo: naive O(T²) projection vs cached O(T) projection
print(f'{"T":>4}  {"no cache(ms)":>11}  {"cached(ms)":>11}  {"speedup":>6}')
print('-' * 40)

for T in [10, 50, 100, 200]:
    d_model, n_heads_t, head_dim = 64, 4, 16
    W_k = np.random.randn(d_model, d_model)

    # No cache: re-project all T tokens at every step
    t0 = time.perf_counter()
    for step in range(T):
        all_x = np.random.randn(step+1, d_model)
        _ = all_x @ W_k
    no_cache_ms = (time.perf_counter() - t0) * 1000

    # With cache: project only the 1 new token at each step
    t0 = time.perf_counter()
    for step in range(T):
        new_x = np.random.randn(1, d_model)
        _ = new_x @ W_k
    cache_ms = (time.perf_counter() - t0) * 1000

    speedup = no_cache_ms / max(cache_ms, 1e-6)
    print(f'{T:>4}  {no_cache_ms:>11.2f}  {cache_ms:>11.2f}  {speedup:>6.1f}×')

## Wrap-up

| Concept | What to remember |
|---|---|
| KV cache | stores past K/V and appends one new slice per step |
| Where the speedup comes from | it avoids order-T repeated K/V linear projections |
| Memory cost | cache size O(T × n_layers × n_heads × head_dim) |
| L86 | sampling strategies — once we have the logits, how do we pick the final token? |

The next lesson (L86) starts on sampling: when the model has handed us a row of logits, how do we pick the final token while balancing "safe" against "lively"?

## ✏️ Closed-book derivation check cell — the cost of a KV cache

**Rule: collapse every cell above and do the derivation below from memory alone.**

**Problem**: at inference time we generate a sequence of length $T$; the model dimension is $d$.

Fill in the table (each cell is the cost of one step, in big-O notation):

| | No cache (recompute every token each step) | With cache (compute only the new token) |
|---|---|---|
| Step $t$: K/V projection cost | $O(t \cdot d^2)$ | $O(___)$ |
| Total projection cost over the sequence | $O(\sum_{t=1}^T t \cdot d^2) = O(___)$ | $O(___)$ |
| Step $t$: attention cost | $O(t \cdot d)$ | $O(___)$ |

1. Explain which row of the table the title's "O(seq²) → O(seq)" refers to
2. A KV cache saves the **projection** cost; does it save attention cost too? Why or why not?

(Fill in the table and answer here...)

In [ ]:
# Check: timing experiment (median of 5 + warmup + larger T, to damp random noise)
import numpy as np, time

np.random.seed(0)
d, T_MAX = 128, 256

def _run_no_cache(xs, Wk, Wv):
    for t in range(1, len(xs) + 1):
        _ = xs[:t] @ Wk
        _ = xs[:t] @ Wv

def _run_cached(xs, Wk, Wv):
    K_cache, V_cache = [], []
    for x in xs:
        K_cache.append(x @ Wk)
        V_cache.append(x @ Wv)

def measure(fn, *args, reps=5):
    fn(*args)  # warmup
    times = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn(*args)
        times.append(time.perf_counter() - t0)
    return float(np.median(times))

xs = np.random.randn(T_MAX, d)
Wk = np.random.randn(d, d) * 0.02
Wv = np.random.randn(d, d) * 0.02

t_no  = measure(_run_no_cache, xs, Wk, Wv)
t_yes = measure(_run_cached,   xs, Wk, Wv)
speedup = t_no / max(t_yes, 1e-9)

print(f"no cache (median):   {t_no*1000:.2f} ms")
print(f"with cache (median): {t_yes*1000:.2f} ms")
print(f"speedup:          {speedup:.1f}×  (T={T_MAX}, d={d})")

# Timing is sensitive to machine load → use a soft check so Run All doesn't fail now and then on a busy machine (the core conclusion doesn't depend on the exact numbers).
if speedup > 1.5:
    print(f"\n✅ KV-cache speedup confirmed ({speedup:.1f}×): the cache turns each step's O(T) repeated projection into O(1).")
else:
    print(f"\n⚠️ Measured {speedup:.1f}× this time (lower values are normal on a busy machine) — in theory no cache is O(T²) and cached is O(T), so the conclusion stands.")

In [ ]:
# ✏️ Self-review for this lesson
l85_review = {
    "kvcache_intuition":         None,  # understand KVCache = append-only store of past K/V, saving order-T repeated projections? True/False
    "complexity_analysis":       None,  # can derive: no cache O(T²d²) vs cached O(Td²)? True/False
    "kvcache_impl":              None,  # KVCache.update correct: assign on first call / concatenate after? True/False
    "consistency_verified":      None,  # with/without-cache outputs matched at atol=1e-9? True/False
    "whiteboard_passed":         None,  # whiteboard challenge (cost analysis) done? True/False
}

unfilled = [k for k, v in l85_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l85_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L85 fully cleared! On to L86: sampling strategies from scratch')

---

→ **Next lesson** · [L86 · Sampling strategies from scratch](L86_sampling.ipynb)

> Next lesson covers **sampling strategies from scratch**: temperature / top-k / top-p in pure NumPy.